# 07 – SQL cross-check of the key results

The main results of questions 2, 3 and 4 were calculated in pandas. Here they are **calculated again, independently, in SQL**, and the two answers are compared. If they match, it is unlikely that a mistake in the Python code changed the conclusions.

The queries are in the [`sql/`](../sql) folder and run with **DuckDB**, which can query CSV files directly without setting up a database server. Each CSV in `data/processed/` is registered as a table (view) so that the SQL reads like a normal database query.

| File | Question | SQL used |
| --- | --- | --- |
| `01_region_summary.sql` | 2: EVs per charger by region | `GROUP BY`, `SUM` |
| `02_deprivation_groups.sql` | 3: five deprivation groups | `JOIN`, `NTILE`, CTE (`WITH`) |
| `03_priority_score.sql` | 4: priority score, top 20 | CTEs, `PERCENT_RANK`, `FILTER` |

In [1]:
import duckdb
import pandas as pd

PROCESSED = "../data/processed/"
SQL = "../sql/"

con = duckdb.connect()
# Register each clean CSV as a table name that the SQL files can use
tables = {
    "la_analysis": "la_analysis_table.csv",
    "deprivation": "deprivation_by_la.csv",
    "housing": "housing_by_la.csv",
    "vehicles": "vehicles_by_la.csv",
}
for name, file in tables.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_csv_auto('{PROCESSED}{file}')")


def run_sql(file):
    query = open(SQL + file).read()
    print(query)
    return con.sql(query).df()

## 1. Region summary (question 2)

In [2]:
region_sql = run_sql("01_region_summary.sql")
region_sql

-- Question 2: private battery electric cars per public charger, by region (July 2026)
SELECT
    region,
    SUM(bev_private) AS bev_private,
    SUM(chargers) AS chargers,
    ROUND(SUM(bev_private) / SUM(chargers), 1) AS bev_private_per_charger
FROM la_analysis
GROUP BY region
ORDER BY bev_private_per_charger;



,region,bev_private,chargers,bev_private_per_charger
0,London,93508.0,31371.0,3.0
1,Scotland,79013.0,12847.0,6.2
2,Wales,37088.0,5910.0,6.3
3,West Midlands,80862.0,10143.0,8.0
4,North East,35121.0,4147.0,8.5
5,South West,86079.0,9232.0,9.3
6,East of England,103606.0,9880.0,10.5
7,South East,167271.0,15373.0,10.9
8,Northern Ireland,13365.0,1156.0,11.6
9,North West,102434.0,8678.0,11.8


In [3]:
# Same calculation in pandas (as in notebook 04)
la = pd.read_csv(PROCESSED + "la_analysis_table.csv")
region_pandas = la.groupby("region")[["bev_private", "chargers"]].sum()
region_pandas = (region_pandas["bev_private"] / region_pandas["chargers"]).round(1)

comparison = region_sql.set_index("region")["bev_private_per_charger"].to_frame("sql")
comparison["pandas"] = region_pandas
print("All regions match:", (comparison["sql"] == comparison["pandas"]).all())

All regions match: True


## 2. Deprivation groups (question 3)

`NTILE(5)` sorts the local authorities by IMD score and splits them into five groups of (almost) equal size, the same as `pd.qcut` in notebook 05.

In [4]:
groups_sql = run_sql("02_deprivation_groups.sql")
groups_sql

-- Question 3: EV adoption and public charging in five deprivation groups (England outside London)

-- Step 1: split all English local authorities into five equal groups by IMD score (1 = least deprived)
WITH england AS (
    SELECT
        la.*,
        d.imd_score,
        NTILE(5) OVER (ORDER BY d.imd_score) AS imd_group
    FROM la_analysis AS la
    JOIN deprivation AS d ON la.la_code = d.la_code
)

-- Step 2: totals per group, leaving London out (as in notebook 05)
SELECT
    imd_group,
    COUNT(*) AS local_authorities,
    ROUND(SUM(bev_private) / SUM(all_vehicles_private) * 100, 1) AS bev_share_pct,
    ROUND(SUM(chargers) / SUM(population_2024) * 100000, 1) AS chargers_per_100k,
    ROUND(SUM(bev_private) / SUM(chargers), 1) AS bev_private_per_charger
FROM england
WHERE region <> 'London'
GROUP BY imd_group
ORDER BY imd_group;



,imd_group,local_authorities,bev_share_pct,chargers_per_100k,bev_private_per_charger
0,1,56,4.0,162.4,13.2
1,2,55,3.4,155.5,11.6
2,3,52,3.1,147.1,10.6
3,4,48,2.7,127.5,9.6
4,5,52,2.6,125.2,8.0


In [5]:
# Same calculation in pandas (as in notebook 05)
deprivation = pd.read_csv(PROCESSED + "deprivation_by_la.csv")
england = la.merge(deprivation[["la_code", "imd_score"]], on="la_code")
england["imd_group"] = pd.qcut(england["imd_score"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
outside_london = england[england["region"] != "London"]
totals = outside_london.groupby("imd_group")[["bev_private", "all_vehicles_private", "chargers", "population_2024"]].sum()
groups_pandas = pd.DataFrame({
    "bev_share_pct": (totals["bev_private"] / totals["all_vehicles_private"] * 100).round(1),
    "chargers_per_100k": (totals["chargers"] / totals["population_2024"] * 100_000).round(1),
    "bev_private_per_charger": (totals["bev_private"] / totals["chargers"]).round(1),
})

columns = ["bev_share_pct", "chargers_per_100k", "bev_private_per_charger"]
print("All groups match:", (groups_sql.set_index("imd_group")[columns] == groups_pandas[columns]).all().all())

All groups match: True


## 3. Priority score (question 4)

The SQL version uses `PERCENT_RANK`, the standard SQL percentile rank. It is calculated as (rank − 1) / (number of rows − 1), while pandas uses rank / number of rows. Both measure the same position on a slightly different scale, so a few scores can differ in the second decimal place, but **the order of local authorities should be the same**.

In [6]:
top20_sql = run_sql("03_priority_score.sql")
top20_sql

-- Question 4: priority score for public charging investment, top 20 (England and Wales)

-- Growth in private battery electric cars, June 2025 -> June 2026
WITH growth AS (
    SELECT
        la_code,
        (SUM(bev_private) FILTER (WHERE quarter = '2026 Q2')
            / SUM(bev_private) FILTER (WHERE quarter = '2025 Q2') - 1) * 100 AS bev_private_growth
    FROM vehicles
    WHERE body_type = 'Cars' AND quarter IN ('2025 Q2', '2026 Q2')
    GROUP BY la_code
),

-- The three indicators; the inner join with housing keeps England and Wales only
indicators AS (
    SELECT
        la.la_code,
        la.la_name,
        la.region,
        la.bev_private_per_charger,
        (h.flats + h.terraced) / h.households * 100 AS no_driveway_share,
        g.bev_private_growth
    FROM la_analysis AS la
    JOIN housing AS h ON la.la_code = h.la_code
    JOIN growth AS g ON la.la_code = g.la_code
),

-- Percentile rank of each indicator (0 = lowest, 1 = highest), like PERCENTRANK in Excel
ranks

,priority_rank,la_name,region,priority_score
0,1,Knowsley,North West,0.85
1,2,Hyndburn,North West,0.84
2,3,Halton,North West,0.83
3,4,Bolton,North West,0.82
4,5,Tameside,North West,0.81
5,6,Pendle,North West,0.77
6,7,North East Lincolnshire,Yorkshire and The Humber,0.77
7,8,North Tyneside,North East,0.77
8,9,Blackburn with Darwen,North West,0.76
9,10,Nuneaton and Bedworth,West Midlands,0.75


In [7]:
# Compare with the ranking saved by notebook 06
ranking = pd.read_csv("../outputs/priority_ranking.csv")
same_order = top20_sql["la_name"].tolist() == ranking["la_name"].head(20).tolist()
largest_difference = (top20_sql["priority_score"] - ranking["priority_score"].head(20)).abs().max()
print("Same top 20 in the same order:", same_order)
print(f"Largest score difference: {largest_difference:.2f}")

Same top 20 in the same order: True
Largest score difference: 0.01


## Result

All three results are the same in SQL and in pandas: the regional ratios, the five deprivation groups and the top 20 priority areas in the same order. The only difference is a rounding-level gap in some priority scores, explained above. The conclusions of the project do not depend on how the numbers were calculated.